To debug and test a graph, you can use static interrupts as breakpoints to step through the graph execution one node at a time. Static interrupts are triggered at defined points either before or after a node executes. You can set these by specifying interrupt_before and interrupt_after when compiling the graph.
- The breakpoints are set during compile time.
- `interrupt_before` specifies the nodes where execution should pause before the node is executed.
- `interrupt_after` specifies the nodes where execution should pause after the node is executed.
- A checkpointer is required to enable breakpoints.
- The graph is run until the first breakpoint is hit.
- The graph is resumed by passing in None for the input. This will run the graph until the next breakpoint is hit.

In [7]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict,Annotated
from operator import add

# 1. Define State
class State(TypedDict):
    data: str
    log: Annotated[list[str], add]

# 2. Define Nodes
def input_node(state: State):
    return {"log": ["Input received"], "data": state["data"]}

def process_node(state: State):
    return {"log": ["Processing data"], "data": state["data"].upper()}

def output_node(state: State):
    return {"log": ["Finalizing"], "data": f"Result: {state['data']}"}

# 3. Build Graph with Breakpoints
builder = StateGraph(State)
builder.add_node("input", input_node)
builder.add_node("process", process_node)
builder.add_node("output", output_node)

builder.add_edge(START, "input")
builder.add_edge("input", "process")
builder.add_edge("process", "output")
builder.add_edge("output", END)

# Checkpointing is REQUIRED for breakpoints
memory = MemorySaver()

# Compile with interrupts before and after the "process" node
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["process"],
    interrupt_after=["process"]
)

config = {"configurable": {"thread_id": "debug_123"}}
initial_input = {"data": "hello world", "log": []}

print("-------------------- FIRST RUN --------------------")
# Stops BEFORE "process" node
for event in graph.stream(initial_input, config,version="v2"):
    print(event)
print("-------------------- STATE -----------------------")
print(graph.get_state(config))
print("-------------------- RESUME 1 --------------------")
# Runs "process" node, then stops AFTER it
for event in graph.stream(None, config,version="v2"):
    print(event)
print("-------------------- STATE -----------------------")
print(graph.get_state(config))
print("-------------------- RESUME 2 --------------------")
# Runs "output" node and finishes
for event in graph.stream(None, config,version="v2"):
    print(event)

-------------------- FIRST RUN --------------------
{'type': 'updates', 'ns': (), 'data': {'input': {'log': ['Input received'], 'data': 'hello world'}}}
{'type': 'updates', 'ns': (), 'data': {'__interrupt__': ()}}
-------------------- STATE -----------------------
StateSnapshot(values={'data': 'hello world', 'log': ['Input received']}, next=('process',), config={'configurable': {'thread_id': 'debug_123', 'checkpoint_ns': '', 'checkpoint_id': '1f14e239-ee0d-6a2f-8001-ac12b6fca54e'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-05-12T16:57:12.155396+00:00', parent_config={'configurable': {'thread_id': 'debug_123', 'checkpoint_ns': '', 'checkpoint_id': '1f14e239-ee0c-6db9-8000-3f51e068e1cf'}}, tasks=(PregelTask(id='694f65ba-c0e7-01db-5c2c-797050cb83d4', name='process', path=('__pregel_pull', 'process'), error=None, interrupts=(), state=None, result=None),), interrupts=())
-------------------- RESUME 1 --------------------
{'type': 'updates', 'ns': (), 'data': {